<a href="https://colab.research.google.com/github/beingAtharun/CN-CD-LAB/blob/main/grounding%20dino%20model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
%pip -q install --upgrade "transformers>=4.51.3" accelerate sentencepiece einops
# Grounding DINO doesn't require bitsandbytes; runs fine in FP16/BF16 on T4.

In [26]:
import torch
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

assert torch.cuda.is_available(), "Enable GPU in Colab: Runtime → Change runtime type → GPU (T4)."
print("GPU:", torch.cuda.get_device_name(0))

MODEL_ID = "IDEA-Research/grounding-dino-base"   # use "grounding-dino-tiny" if you want more speed

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForZeroShotObjectDetection.from_pretrained(MODEL_ID).to("cuda")

print("Loaded:", MODEL_ID)

GPU: Tesla T4


preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

The image processor of type `GroundingDinoImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/933M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1206 [00:00<?, ?it/s]

Loaded: IDEA-Research/grounding-dino-base


In [28]:
import re, math, json
from typing import List, Dict, Any
from PIL import Image, ImageDraw, ImageFont

def preprocess_queries(categories: List[str]) -> str:
    """
    Grounding DINO expects lowercased queries, each ending with '.' and separated by '. '.
    e.g., ['milk','soft drinks'] -> 'milk. soft drinks.'
    """
    cats = []
    for c in categories:
        c = c.strip().lower()
        if not c.endswith("."):
            c += "."
        cats.append(c)
    # Deduplicate while keeping order
    seen, out = set(), []
    for c in cats:
        if c not in seen:
            out.append(c)
            seen.add(c)
    return " ".join(out)

def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    if inter <= 0:
        return 0.0
    areaA = max(0, boxA[2]-boxA[0]) * max(0, boxA[3]-boxA[1])
    areaB = max(0, boxB[2]-boxB[0]) * max(0, boxB[3]-boxB[1])
    union = areaA + areaB - inter
    return inter / max(union, 1e-6)

def nms_merge(instances: List[Dict[str, Any]], iou_thresh: float = 0.5) -> List[Dict[str, Any]]:
    """
    Class-aware NMS: keeps a box if it doesn't highly overlap (IoU >= iou_thresh)
    with a kept box of the same label. Sort by score*area to favor larger, confident boxes.
    Each item: {"label": str, "score": float, "box_px": [x1,y1,x2,y2]}
    """
    def area(b): return max(0, b[2]-b[0]) * max(0, b[3]-b[1])
    instances = sorted(instances, key=lambda d: d["score"] * area(d["box_px"]), reverse=True)
    kept = []
    for d in instances:
        if all(iou(d["box_px"], k["box_px"]) < iou_thresh or d["label"] != k["label"] for k in kept):
            kept.append(d)
    return kept

def clip_box(x1: float, y1: float, x2: float, y2: float, W: int, H: int) -> List[int]:
    """
    Clamp box to image bounds and ensure proper ordering.
    """
    x1 = max(0, min(W-1, int(round(x1))))
    y1 = max(0, min(H-1, int(round(y1))))
    x2 = max(0, min(W-1, int(round(x2))))
    y2 = max(0, min(H-1, int(round(y2))))
    if x2 < x1: x1, x2 = x2, x1
    if y2 < y1: y1, y2 = y2, y1
    return [x1, y1, x2, y2]

def to_1000(box_px: List[int], W: int, H: int) -> List[int]:
    """
    Convert pixel box [x1,y1,x2,y2] to a 0..1000 coordinate system relative to (W,H).
    Ensures X1<=X2 and Y1<=Y2.
    """
    x1, y1, x2, y2 = box_px
    X1 = max(0, min(1000, int(round((x1 / W) * 1000))))
    Y1 = max(0, min(1000, int(round((y1 / H) * 1000))))
    X2 = max(0, min(1000, int(round((x2 / W) * 1000))))
    Y2 = max(0, min(1000, int(round((y2 / H) * 1000))))
    if X2 < X1: X1, X2 = X2, X1
    if Y2 < Y1: Y1, Y2 = Y2, Y1
    return [X1, Y1, X2, Y2]

def from_tile_box_to_full(bb: List[float], x0: int, y0: int) -> List[float]:
    """
    Convert a tile-local pixel box [x1,y1,x2,y2] to full-image coords by offset (x0,y0).
    """
    x1, y1, x2, y2 = bb
    return [x0 + x1, y0 + y1, x0 + x2, y0 + y2]

In [29]:
import torch

@torch.no_grad()
def gdino_detect(image: Image.Image,
                 categories: List[str],
                 box_threshold: float = 0.35,
                 text_threshold: float = 0.25) -> List[Dict[str, Any]]:
    """
    Returns list of dicts: {label, score, box_px=[x1,y1,x2,y2]}
    """
    W, H = image.size
    text = preprocess_queries(categories)  # lowercased, period-separated
    inputs = processor(images=image, text=text, return_tensors="pt").to("cuda")

    # Forward
    outputs = model(**inputs)

    # Post-process to image size
    results = processor.post_process_grounded_object_detection(
        outputs=outputs,
        input_ids=inputs.input_ids,
        box_threshold=box_threshold,
        text_threshold=text_threshold,
        target_sizes=[(H, W)]  # (height, width)
    )[0]

    # results format: boxes (x1,y1,x2,y2), labels (text), scores
    out = []
    for bx, lab, sc in zip(results["boxes"].tolist(),
                           results["labels"],
                           results["scores"].tolist()):
        out.append({"label": lab, "score": float(sc), "box_px": clip_box(*bx, W, H)})
    return out

In [30]:
def run_gdino_tiled(image: Image.Image,
                    categories: List[str],
                    grid=(4,3),
                    overlap=0.2,
                    box_threshold=0.35,
                    text_threshold=0.25,
                    iou_merge=0.5) -> Dict[str, Any]:
    """
    Tile the image to improve recall for small/repetitive items, then merge with NMS.
    Builds a JSON similar to your earlier structure:
    {
      "product_categories": [...],
      "empty_shelves": 0,     # optional heuristic; set 0 by default here
      "products": [
         {"product_name":"item","category":"chips","bounding_box":[X1,Y1,X2,Y2]}
      ]
    }
    """
    W, H = image.size
    gx, gy = grid
    tile_w = math.ceil(W / gx)
    tile_h = math.ceil(H / gy)
    ox = int(overlap * tile_w)
    oy = int(overlap * tile_h)

    all_instances = []
    for yi in range(gy):
        for xi in range(gx):
            x0 = max(0, xi*tile_w - (ox if xi>0 else 0))
            y0 = max(0, yi*tile_h - (oy if yi>0 else 0))
            x1 = min(W, (xi+1)*tile_w + (ox if xi<gx-1 else 0))
            y1 = min(H, (yi+1)*tile_h + (oy if yi<gy-1 else 0))
            crop = image.crop((x0,y0,x1,y1))

            dets = gdino_detect(crop, categories, box_threshold, text_threshold)
            for d in dets:
                full_box = from_tile_box_to_full(d["box_px"], x0, y0)
                d2 = {"label": d["label"], "score": d["score"], "box_px": clip_box(*full_box, W, H)}
                all_instances.append(d2)

    merged = nms_merge(all_instances, iou_thresh=iou_merge)

    final_products = [{
        "product_name": "item",                 # Grounding DINO doesn't return product name strings
        "category":     m["label"],
        "score":        round(m["score"], 4),
        "bounding_box": to_1000(m["box_px"], W, H)
    } for m in merged]

    # We keep empty_shelves as 0 by default here; see note below to estimate it.
    return {
        "product_categories": sorted(set([c.strip().lower() for c in categories])),
        "empty_shelves": 0,
        "products": final_products
    }

In [32]:
def gdino_postprocess(outputs, inputs, image_size_wh, box_threshold=0.35, text_threshold=0.25):
    """
    Backward-compatible wrapper for Grounding DINO post-processing.
    Tries newer Transformers (with box/text threshold kwargs). If unavailable,
    calls older signature and filters by score manually.
    """
    W, H = image_size_wh
    target_sizes = [(H, W)]  # HF expects (height, width)

    try:
        # Newer Transformers: supports explicit thresholds as kwargs
        res = processor.post_process_grounded_object_detection(
            outputs=outputs,
            input_ids=inputs.input_ids,
            box_threshold=box_threshold,
            text_threshold=text_threshold,
            target_sizes=target_sizes
        )[0]
        return res
    except TypeError:
        # Older signature: no threshold kwargs -> call without them and filter manually
        res = processor.post_process_grounded_object_detection(
            outputs, inputs.input_ids, target_sizes=target_sizes
        )[0]

        boxes = res.get("boxes", [])
        scores = res.get("scores", [])
        labels = res.get("labels", [])

        # Convert tensors -> lists if needed
        try:
            boxes = boxes.tolist()
        except Exception:
            pass
        try:
            scores = scores.tolist()
        except Exception:
            pass

        keep = [i for i, s in enumerate(scores) if s >= box_threshold]
        filtered = {
            "boxes": [boxes[i] for i in keep],
            "scores": [scores[i] for i in keep],
            "labels": [labels[i] for i in keep] if isinstance(labels, list) else labels
        }
        return filtered

In [33]:
import torch

@torch.no_grad()
def gdino_detect(image: Image.Image,
                 categories: List[str],
                 box_threshold: float = 0.35,
                 text_threshold: float = 0.25) -> List[Dict[str, Any]]:
    """
    Returns list of dicts: {label, score, box_px=[x1,y1,x2,y2]} in image pixels.
    """
    W, H = image.size
    text = preprocess_queries(categories)  # lowercased, dot-separated
    inputs = processor(images=image, text=text, return_tensors="pt").to("cuda")

    # Forward
    outputs = model(**inputs)

    # Backward-compatible post-processing
    res = gdino_postprocess(
        outputs, inputs, (W, H),
        box_threshold=box_threshold,
        text_threshold=text_threshold
    )

    out = []
    boxes = res.get("boxes", [])
    labels = res.get("labels", [])
    scores = res.get("scores", [])

    # Convert tensors -> lists if needed
    try: boxes = boxes.tolist()
    except Exception: pass
    try: scores = scores.tolist()
    except Exception: pass
    if not isinstance(labels, list):
        try: labels = labels.tolist()
        except Exception: pass

    for bx, lab, sc in zip(boxes, labels, scores):
        out.append({
            "label": lab if isinstance(lab, str) else str(lab),
            "score": float(sc),
            "box_px": clip_box(*bx, W, H)
        })
    return out

In [34]:
from google.colab import files
uploaded = files.upload()
img_path = list(uploaded.keys())[0]
image = Image.open(img_path).convert("RGB")

categories = ["chips", "soft drinks", "milk", "biscuits", "chocolates"]

result = run_gdino_tiled(
    image,
    categories=categories,
    grid=(4,3),
    overlap=0.2,
    box_threshold=0.35,
    text_threshold=0.25,
    iou_merge=0.5
)

print(json.dumps(result, indent=2, ensure_ascii=False))

Saving image (2).png to image (2) (2).png


/usr/local/lib/python3.12/dist-packages/transformers/models/grounding_dino/processing_grounding_dino.py:96: FutureWarning: The key `labels` is will return integer ids in `GroundingDinoProcessor.post_process_grounded_object_detection` output since v4.51.0. Use `text_labels` instead to retrieve string object names.
  warnings.warn(self.message, FutureWarning)


{
  "product_categories": [
    "biscuits",
    "chips",
    "chocolates",
    "milk",
    "soft drinks"
  ],
  "empty_shelves": 0,
  "products": [
    {
      "product_name": "item",
      "category": "soft drinks",
      "score": 0.372,
      "bounding_box": [
        203,
        619,
        549,
        776
      ]
    },
    {
      "product_name": "item",
      "category": "chocolates",
      "score": 0.3547,
      "bounding_box": [
        203,
        619,
        549,
        776
      ]
    },
    {
      "product_name": "item",
      "category": "chocolates",
      "score": 0.4376,
      "bounding_box": [
        454,
        640,
        762,
        773
      ]
    },
    {
      "product_name": "item",
      "category": "biscuits",
      "score": 0.4191,
      "bounding_box": [
        454,
        637,
        759,
        734
      ]
    },
    {
      "product_name": "item",
      "category": "soft drinks",
      "score": 0.389,
      "bounding_box": [
        454,
  

In [35]:
W, H = image.size
vis = image.copy()
draw = ImageDraw.Draw(vis)
try:
    font = ImageFont.truetype("DejaVuSans.ttf", size=max(12, W//85))
except:
    font = None

for p in result.get("products", []):
    X1,Y1,X2,Y2 = p["bounding_box"]
    x1 = int(round((X1/1000)*W)); y1 = int(round((Y1/1000)*H))
    x2 = int(round((X2/1000)*W)); y2 = int(round((Y2/1000)*H))
    draw.rectangle([x1,y1,x2,y2], outline="lime", width=3)
    label = f"{p.get('category','')}: {p.get('score',0):.2f}"
    try:
        draw.text((x1, max(0, y1-18)), label, fill="yellow",
                  font=font, stroke_width=2, stroke_fill="black")
    except:
        draw.text((x1, max(0, y1-18)), label, fill="yellow")

out_path = img_path.rsplit(".",1)[0] + "_gdino_viz.jpg"
vis.save(out_path)
out_path

'image (2) (2)_gdino_viz.jpg'